In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/deep-past-initiative-machine-translation/train.csv
/kaggle/input/deep-past-initiative-machine-translation/test.csv
/kaggle/input/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/deep-past-initiative-machine-translation/resources.csv


In [2]:
INPUT_DIR = "/kaggle/input/deep-past-initiative-machine-translation/"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")
publications_df = pd.read_csv(f"{INPUT_DIR}/publications.csv")
dict_link_df = pd.read_csv(f"{INPUT_DIR}/OA_Lexicon_eBL.csv")
dict_df = pd.read_csv(f"{INPUT_DIR}/eBL_Dictionary.csv")
alignment_df = pd.read_csv(f"{INPUT_DIR}/Sentences_Oare_FirstWord_LinNum.csv")
resources_df = pd.read_csv(f"{INPUT_DIR}/resources.csv")
published_df = pd.read_csv(f"{INPUT_DIR}/published_texts.csv")

Use Dict_df, along with a brute force collection of all words found to get a list of all of the words.

Remove all gaps and treat them as seperate sentences.

use train_df, test_df, published_df as data for the word2vec model.
If there's a way I can get the counts of each word I would like to know.

https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html

In [3]:
# For loop through train_df, test_df, published_df transliterations all as one big list.
# Add each sentence to the thing. 
# If a sentence has one or more gaps/big gaps (<big_gap>, <gap>), split on those

sentences = []
for df in [train_df, test_df, published_df]:
    for col in ["transliteration"]:
        for sentence in df[col].dropna():
            sentences.append(sentence)
            if "<big_gap>" in sentence:
                sentences.extend(sentence.split("<big_gap>"))
            if "<gap>" in sentence:
                sentences.extend(sentence.split("<gap>"))
sentences = list(set(sentences))  # Remove duplicates
sentences = [s.strip() for s in sentences if s.strip()]  # Remove empty


In [4]:
from gensim.test.utils import datapath
from gensim import utils

class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __iter__(self):
        # corpus_path = datapath('lee_background.cor')
        # for line in open(corpus_path):
        for line in sentences:
            # assume there's one document per line, tokens separated by whitespace
            yield utils.simple_preprocess(line)

In [5]:
import gensim.models

sentencesCorpus = MyCorpus()
model = gensim.models.Word2Vec(sentences=sentencesCorpus)

In [7]:
for index, word in enumerate(model.wv.index_to_key):
    if index == 10:
        break
    print(f"word #{index}/{len(model.wv.index_to_key)} is {word}")

word #0/367 is ma
word #1/367 is na
word #2/367 is ša
word #3/367 is šu
word #4/367 is ni
word #5/367 is lá
word #6/367 is ta
word #7/367 is kù
word #8/367 is babbar
word #9/367 is ku


In [ ]:
import tempfile

with tempfile.NamedTemporaryFile(prefix='gensim-model-', delete=False) as tmp:
    temporary_filepath = tmp.name
    model.save(temporary_filepath)
    #
    # The model is now safely stored in the filepath.
    # You can copy it to other machines, share it with others, etc.
    #
    # To load a saved model:
    #
    new_model = gensim.models.Word2Vec.load(temporary_filepath)